# Superstore — Shipping & Regional Analysis

A reproducible Python companion to the Superstore Sales Power BI project. The analysis focuses on regional profitability, shipping-mode mix, delivery time, late Standard Class shipments, and discount pressure.

**Dataset:** Sample Superstore · 9,994 order lines · 2014–2017

> This notebook is descriptive: it identifies patterns and business hypotheses without claiming causal effects from observational data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = '../data/Sample - Superstore.csv'
df = pd.read_csv(DATA_PATH, encoding='latin1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df['Days to Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

print(f'Rows: {len(df):,}')
print(f'Date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
df.head()

## 1. Regional performance

Which regions combine scale with healthy profitability?

In [ ]:
regional = (df.groupby('Region')
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Orders=('Order ID','nunique'), Customers=('Customer ID','nunique'))
    .assign(Margin=lambda x: x['Profit'] / x['Sales'] * 100)
    .sort_values('Sales', ascending=False))
regional.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
regional.sort_values('Sales')['Sales'].plot(kind='barh', ax=ax)
ax.set_title('Sales by Region')
ax.set_xlabel('Sales ($)')
plt.tight_layout()
plt.show()

### Interpretation
The project report identifies Central as the key profitability issue: about $501K sales at 7.92% margin versus 14.94% in West. Central is a large market with comparatively weak return.

## 2. Shipping-mode mix by region

Is Central structurally different in its shipping mix?

In [ ]:
ship_mix = (df.groupby(['Region','Ship Mode'])['Sales'].sum().unstack(fill_value=0))
ship_mix.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ship_mix.plot(kind='bar', ax=ax)
ax.set_title('Sales by Shipping Mode and Region')
ax.set_xlabel('Region')
ax.set_ylabel('Sales ($)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Ship Mode')
plt.tight_layout()
plt.show()

Standard Class is the dominant shipping mode across regions. The report describes the shipping mix as broadly proportional, so the Central margin gap should not automatically be attributed to shipping-mode composition.

## 3. Average delivery time

Do regions have materially different delivery performance?

In [ ]:
ship_days = df.groupby(['Ship Mode','Region'])['Days to Ship'].mean().unstack()
ship_days.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ship_days.plot(kind='bar', ax=ax)
ax.set_title('Average Days to Ship by Mode and Region')
ax.set_xlabel('Ship Mode')
ax.set_ylabel('Average calendar days')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Region')
plt.tight_layout()
plt.show()

The supplied report states that Standard Class averages roughly five days across all four regions. The near-identical values weaken logistics as the main explanation for Central's margin problem.

## 4. Late Standard Class shipments

The project defines a late Standard Class shipment as taking more than five calendar days.

In [ ]:
late_standard = (df.loc[(df['Ship Mode']=='Standard Class') & (df['Days to Ship']>5)]
    .groupby('Region').size().sort_values(ascending=False)
    .rename('Late Standard Class Shipments'))
late_standard.to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
late_standard.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Late Standard Class Shipments (>5 days)')
ax.set_xlabel('Order lines')
plt.tight_layout()
plt.show()

Raw late-shipment counts should be interpreted alongside regional order volume; the report notes that West has the highest count largely because it is the largest region.

## 5. Profit margin by shipping mode and region

In [ ]:
margin_matrix = (df.groupby(['Region','Ship Mode'])[['Sales','Profit']].sum()
    .assign(Margin=lambda x: x['Profit']/x['Sales']*100)['Margin'].unstack())
margin_matrix.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
im = ax.imshow(margin_matrix.values, aspect='auto')
ax.set_title('Profit Margin by Region and Shipping Mode')
ax.set_xticks(range(len(margin_matrix.columns)), margin_matrix.columns, rotation=25, ha='right')
ax.set_yticks(range(len(margin_matrix.index)), margin_matrix.index)
for i in range(margin_matrix.shape[0]):
    for j in range(margin_matrix.shape[1]):
        ax.text(j, i, f'{margin_matrix.iloc[i,j]:.1f}%', ha='center', va='center')
ax.set_xlabel('Ship Mode')
ax.set_ylabel('Region')
plt.colorbar(im, ax=ax, label='Margin (%)')
plt.tight_layout()
plt.show()

## 6. Discount pressure

The broader Power BI analysis finds that discounts above 30% are loss-generating and identifies discount policy as the stronger commercial hypothesis behind margin leakage.

In [ ]:
discount_summary = (df.assign(Discount_Band=pd.cut(df['Discount'], bins=[-0.001,0,0.10,0.20,0.30,0.40,0.50,1.0], labels=['0%','1–10%','11–20%','21–30%','31–40%','41–50%','51%+']))
    .groupby('Discount_Band', observed=False)
    .agg(Sales=('Sales','sum'), Profit=('Profit','sum'), Order_Lines=('Row ID','count'))
    .assign(Margin=lambda x: x['Profit']/x['Sales']*100))
discount_summary.round(2)

## 7. Business conclusions

- Central is the main regional profitability concern.
- Standard Class dominates sales across regions.
- Delivery times are broadly similar across regions.
- Late-shipment counts should be normalized for regional scale.
- The stronger business hypothesis is commercial margin leakage, particularly pricing and discount policy.
- The Power BI report recommends a 25% discount cap, targeted repricing of Tables and Bookcases, and a Central-region profitability program.

### Analytical caveat
These are observational patterns, not causal estimates. Pricing or discount interventions should be validated with controlled experiments or quasi-experimental analysis before attributing incremental profit to a policy.